In [49]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Data Pre-Processing

1. MNIST images mostly consist of grayscale images; hence their dimensions are [rows * columns * 1], their is only 1 channel; hence depth == 1. 
2. We are going to convert them to tensors so it's easy to pass them to Pytorch functions.
3. Let's also normalize the - before that; lets find their mean and std. 

In [2]:
# Compute MNIST mean and std from the dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
loader = DataLoader(train_dataset, batch_size=60000)
images, _ = next(iter(loader))
mean = images.mean().item()
std = images.std().item()
print(f"Computed mean: {mean:.4f}, std: {std:.4f}")

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.55MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 348kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.37MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.3MB/s]


Computed mean: 0.1307, std: 0.3081


In [4]:
# 1. Pipeline: Convert PIL images to Tensors and Normalize (Mean=0.1307, Std=0.3081)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])

## Split the data into train, test and validation datasets

In [22]:
from torch.utils.data import random_split
train_size = int(len(train_dataset) * 0.7)
test_size = int(len(train_dataset) * 0.2)
val_size = len(train_dataset) - train_size - test_size
train_subset, test_subset, val_subset = random_split(train_dataset, [train_size, test_size, val_size])

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64)
test_loader = DataLoader(test_subset,  batch_size=64)
# we don't need loader for test since we can drive inference on all of them together.

# Construct the Neural Net

Also, 
> nn.Conv2d(in_channels, out_channels, kernel_size)

1. Each image in MNIST dataset has shape 1 * 28 * 28. 
2. Next, in the first layer, we have 32 filters each with kernel size 3*3. Hence the output of this layer will be: 32 * 26 * 26
3. Next we pipe these 32 outputs of 1 * 26 * 26 into 64 filters of kernel size 3 * 3 * 32. Mind you that 1 filter in the last layer was a 3*3 filter, and there were 32 of them, so it was a  2D filter. 
4. In this layer, the filter itself is 3D. 3 * 3 * 32.
 > So each filter has 3 * 3 * 32 = 288 weights. It slides over the 26x26 region:
 > At each position, it takes a 3x3 patch from each of the 32 channels, multiplies by its 288 weights, sums → one number.
 > This repeats across the 26x26 grid → one 24x24 output map per filters.
 > With 64 filters → 64 such maps.
4. Outputs are then: 64 * 24 * 24. (24 = 26 - 3 + 1)
5. Next we have a linear layer, which connects all the 64 * 24 * 24 = 36864 neurons to 128 neurons.
6. Last layer connects the 128 neurons to 10 output neurons, since these are digits we need 10 classes to classify them.

In [7]:
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        # 1. Conv layer: 1 input channel, 32 filters, kernel 3
        self.conv1 = nn.Conv2d(1, 32, 3)
        # 2. Conv layer: 32 -> 64 filters
        self.conv2 = nn.Conv2d(32, 64, 3)
        # 3. Fully connected: flatten 64*24*24 -> 128
        self.fc1 = nn.Linear(36864, 128)
        # 4. Output: 128 -> 10 classes
        self.fc2 = nn.Linear(128, 10)

# add all the activation functions
    def forward(self, x):

        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.relu(self.conv2(x))

        """
        x.view(x.size(0), -1):

        x.size(0) = batch size (e.g., 64)
        -1 = "figure out the rest" → 64*24*24 = 36864
        So it reshapes (batch, 64, 24, 24) → (batch, 36864) without copying data.
        """

        x = x.view(x.size(0), -1)  # flatten

        x = nn.functional.relu(self.fc1(x))

        # counter-intuitive:
        """
        Since you would think since we are solving a classification problem
        using logits would be probably best: hinting at using softmax as the activation function.
        
        But as so it turns out: we shall use the CrossEntropyLoss as the loss function
        which expects rw logits since it internally applies softmax. 

        If we use `softmax` in the forward pass, we will need to use the NLLLoss: Negative Log Likelihood instead. 
        It takes log-probabilities (from log_softmax) and compares to target class.
        CrossEntropyLoss = log_softmax + NLLLoss combined.
        """
        x = self.fc2(x)

        return x

# Define loss function and Optimizer, and Initialize Model

In [8]:
MNISTModel = MNISTNet()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(MNISTModel.parameters(), lr=0.01, momentum=0.9)

# Training Loop

In [20]:
# define number of ecpohs
n_epochs = 10

# create variables to store training and validation loss
training_loss, val_loss = [], []

for i in range(n_epochs):

    # batching
    for images, digits in train_loader:

        # zero the gradients
        optimizer.zero_grad()

        # forward pass
        predDigit = MNISTModel(images)

        # compute loss function
        loss = criterion(predDigit, digits)

        # compute gradients
        loss.backward()

        # update weights
        optimizer.step()

    # store training loss
    training_loss.append(loss.item())

    # store validation loss
    with torch.no_grad():

        for val_images, val_digits in val_loader:

            # forward pass
            valDigit = MNISTModel(val_images)

            # compute loss
            valLoss = criterion(valDigit, val_digits)

        # append validation loss
        val_loss.append(valLoss.item())

# Inference

In [47]:
with torch.no_grad():

    for test_images, test_digits in test_loader:

        # forward pass
        y_pred = MNISTModel(test_images)

        # find maximum probability candidate
        # compare to labels:
        accuracy = len(np.where(test_digits == y_pred.argmax(dim=1))[0])*100/len(test_digits)

In [48]:
accuracy

100.0